In [10]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

In [11]:

df = pd.read_csv('Cleaned_Karachi_100_Data.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f"Shape      : {df.shape}")
print(f"\nMissing values per column:")
print(df.isnull().sum().to_string())
df.head(3)

Shape      : (2514, 9)

Missing values per column:
Date                 0
Price                0
Open                 0
High                 0
Low                  0
Vol.              1196
Change %             0
Daily_Return         1
Volatility_30d      30


,Date,Price,Open,High,Low,Vol.,Change %,Daily_Return,Volatility_30d
0,2016-01-01,33228.95,32841.49,33256.85,32828.36,73480000.0,1.26,NaN,NaN
1,2016-01-04,33009.13,33265.58,33304.40,32968.63,68940000.0,-0.66,-0.661532,NaN
2,2016-01-05,32983.47,32804.32,33027.33,32515.05,99320000.0,-0.08,-0.077736,NaN


In [12]:
# TREND FEATURES
close = df['Price']
high  = df['High']
low   = df['Low']
vol   = df['Vol.']

# Simple Moving Averages
for w in [10, 20, 50, 200]:
    df[f'SMA_{w}'] = close.rolling(w).mean()

# Exponential Moving Averages
for w in [10, 20, 50]:
    df[f'EMA_{w}'] = close.ewm(span=w, adjust=False).mean()

# Crossover signals (binary flags)

#Golden_Cross	—	Binary: 1 when SMA_50 > SMA_200
#EMA_crossover	—	Binary: 1 when EMA_10 > EMA_20
df['Golden_Cross']   = (df['SMA_50'] > df['SMA_200']).astype(int)
df['EMA_crossover']  = (df['EMA_10'] > df['EMA_20']).astype(int)

gc_pct = df['Golden_Cross'].mean() * 100
print(f"Days in Golden Cross (bull) regime : {gc_pct:.1f}%")
print(f"Days in Death Cross  (bear) regime : {100 - gc_pct:.1f}%")


Days in Golden Cross (bull) regime : 52.5%
Days in Death Cross  (bear) regime : 47.5%


In [13]:
# MOMENTUM FEATURES

### RSI — Relative Strength Index
# Compares the average of up-day closes to down-day closes over the last 14 days scaled to 0–100.

# > 70 :  "Overbought": recent gains have been unusually strong; a pullback becomes more likely
# < 30 :  "Oversold": recent losses have been unusually severe; a bounce becomes more likely

def compute_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

df['RSI_14'] = compute_rsi(close, 14)
df['RSI_OB'] = (df['RSI_14'] > 70).astype(int)   # overbought flag
df['RSI_OS'] = (df['RSI_14'] < 30).astype(int)   # oversold flag

# MACD — Moving Average Convergence Divergence
# MACD = EMA(12) − EMA(26). Positive means recent price is rising faster than the medium-term average.
# MACD_Signal = EMA(9) of MACD. The smoothed trend of MACD.
# MACD_Hist = MACD − Signal. This is the most useful form — it measures the acceleration/deceleration of momentum. When the histogram turns from negative to positive, momentum is shifting upward.
ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
df['MACD']        = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']

# Rate of Change
# ROC_n = (Price_t − Price_{t−n}) / Price_{t−n} × 100. Simple percentage price change over n days. Stationary and directly interpretable.
df[f'ROC_5'] = close.pct_change(5) * 100


print(f"RSI overbought days (>70): {df['RSI_OB'].sum()}  ({df['RSI_OB'].sum()/len(df)*100:.1f}% of sample)")
print(f"RSI oversold days  (<30): {df['RSI_OS'].sum()}  ({df['RSI_OS'].sum()/len(df)*100:.1f}% of sample)")

RSI overbought days (>70): 475  (18.9% of sample)
RSI oversold days  (<30): 98  (3.9% of sample)


In [14]:
# VOLATILITY FEATURES
# Bollinger Bands (20-day, ±2 std)
# 20-day SMA with bands at ±2 standard deviations:
# BB_Width = (Upper − Lower) / Middle  measures how wide the bands are. Narrow = low volatility (calm market). Wide = high volatility.
# BB_Pct = (Close − Lower) / (Upper − Lower)  where is today's price within the band? 0 = at the bottom band, 1 = at the top band, 0.5 = at the midpoint.

bb_mid = close.rolling(20).mean()
bb_std = close.rolling(20).std()
df['BB_Upper']  = bb_mid + 2 * bb_std
df['BB_Lower']  = bb_mid - 2 * bb_std
df['BB_Middle'] = bb_mid
df['BB_Width']  = (df['BB_Upper'] - df['BB_Lower']) / bb_mid
df['BB_Pct']    = (close - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower']).replace(0, np.nan)

# ATR — Average True Range
# True Range = max(High−Low, |High−PrevClose|, |Low−PrevClose|)
# ATR_14 = 14-day exponential average of True Range
# ATR_Pct = ATR_14 / Price × 100

tr = pd.concat([
    high - low,
    (high - close.shift(1)).abs(),
    (low  - close.shift(1)).abs()
], axis=1).max(axis=1)
atr_14 = tr.ewm(span=14, adjust=False).mean()
df['ATR_Pct'] = atr_14 / close * 100

# Historical Volatility (annualised)
# HV = std(log returns over 20 days) × √252 × 100 — annualised volatility in % terms.
log_ret = np.log(close / close.shift(1))
df['HV_20'] = log_ret.rolling(20).std() * np.sqrt(252) * 100

In [15]:
# VOLUME FEATURES\
# Vol_Ratio: Today's volume / 20-day average.
vol_sma_20 = vol.rolling(20).mean()
df['Vol_Ratio'] = vol / vol_sma_20

In [16]:
df.head()

,Date,Price,Open,High,Low,Vol.,Change %,Daily_Return,Volatility_30d,SMA_10,...,MACD_Hist,ROC_5,BB_Upper,BB_Lower,BB_Middle,BB_Width,BB_Pct,ATR_Pct,HV_20,Vol_Ratio
0,2016-01-01,33228.95,32841.49,33256.85,32828.36,73480000.0,1.26,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.289508,NaN,NaN
1,2016-01-04,33009.13,33265.58,33304.40,32968.63,68940000.0,-0.66,-0.661532,NaN,NaN,...,-14.028399,NaN,NaN,NaN,NaN,NaN,NaN,1.260643,NaN,NaN
2,2016-01-05,32983.47,32804.32,33027.33,32515.05,99320000.0,-0.08,-0.077736,NaN,NaN,...,-23.691323,NaN,NaN,NaN,NaN,NaN,NaN,1.300493,NaN,NaN
3,2016-01-06,32968.26,33036.63,33123.53,32930.37,61270000.0,-0.05,-0.046114,NaN,NaN,...,-29.390615,NaN,NaN,NaN,NaN,NaN,NaN,1.205733,NaN,NaN
4,2016-01-07,32682.50,32964.65,32964.65,32652.39,79280000.0,-0.87,-0.866773,NaN,NaN,...,-49.505693,NaN,NaN,NaN,NaN,NaN,NaN,1.182970,NaN,NaN


In [17]:
final_cols = [
    'Date', 'Price', 'Daily_Return',
    'Golden_Cross', 'EMA_crossover',
    'RSI_14', 'MACD_Hist', 'ROC_5',
    'BB_Width', 'BB_Pct', 'ATR_Pct', 'HV_20',
    'Vol_Ratio'
]

df_final = df[final_cols]
df_final.to_csv('KSE100_technical_features.csv', index=False)